List all collections
=======================
curl http://localhost:6333/collections



In [ ]:
import sys, os

# Get the directory where the notebook is running
notebook_dir = os.getcwd()

# Go up three levels: models → src → backend → project root
project_root = os.path.abspath(
    os.path.join(notebook_dir, "..")
)

# Add project root to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)
print("src in path:", any("src" in p for p in sys.path))

In [ ]:
#Connect to Qdrant
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(
    url="http://localhost:6333",  # your docker container
)


In [ ]:
#List collections
qdrant.get_collections()

In [ ]:
#Check if a collection exists
qdrant.collection_exists("documents")

In [ ]:
#Generate embeddings
import os
from services.rag.document_handler import documentHandler

filename = "Microsoft.txt"
def file_finder(filename, base_dir="files"):
    for root, dirs, files in os.walk(base_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

result = file_finder(filename)
if result:
    print(f"Found: {result}")
else:
    print("File not found.")

doc = documentHandler(file_path=result, file_name=filename)
text = ["When was microsoft founded"]
embedding = doc.embed_batch(text)
query_vector = embedding[0]

In [ ]:
print(query_vector)

In [ ]:
#Vector search
result = qdrant.query_points(
    collection_name="documents",
    query=query_vector,   # just pass the list of floats directly
    limit=5
)

for point in result.points:
    print(point.payload, point.score)

In [ ]:
#Filtered search
result = qdrant.query_points(
    collection_name="documents",
    query=query_vector,
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="source",
                match=models.MatchValue(value=filename)
            )
        ]
    ),
    limit=5
)

for point in result.points:
    print(point.payload, point.score)

In [ ]:
#retrieve ids
points, next_page = qdrant.scroll(
    collection_name="documents",
    limit=10
)

for p in points:
    print(p.id, p.payload)


In [ ]:
#Retrieve a point
id = "05808056-cb51-4208-a415-e2110d54a898"
qdrant.retrieve("documents", ids=[id])


In [ ]:
#Delete points
id = ""
qdrant.delete(
    collection_name="documents",
    points_selector=models.PointIdsList(points=[id])
)


In [ ]:
#Delete a collection
qdrant.delete_collection("documents")
